In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.metrics import make_scorer, f1_score


In [13]:
# ==============================
# 2. Load Cleaned Dataset
# ==============================

data = pd.read_csv("../data/processed/patient_state_clean.csv")

print("Dataset shape:", data.shape)
data.head()


Dataset shape: (2899, 12)


,SEQN,Age,Gender,BMI,Systolic_BP,Diastolic_BP,Glucose,HbA1c,Insulin,Total_Cholesterol,HDL_Cholesterol,Metabolic_Risk
0,93708.0,66.0,1.0,23.7,141.000000,77.000000,122.0,6.2,9.72,209.0,88.0,1
1,93711.0,56.0,0.0,21.3,101.333333,66.666667,107.0,5.7,5.28,238.0,72.0,1
2,93717.0,22.0,0.0,24.5,118.666667,65.333333,91.0,5.1,3.94,213.0,53.0,0
3,93718.0,45.0,0.0,22.0,131.333333,90.000000,89.0,5.7,4.89,152.0,63.0,1
4,93719.0,13.0,1.0,26.0,101.333333,64.000000,86.0,5.0,10.94,97.0,46.0,0


In [14]:
FEATURES = [
    "Age", "Gender", "BMI",
    "Systolic_BP", "Diastolic_BP",
    "Glucose", "Insulin",
    "Total_Cholesterol", "HDL_Cholesterol"
]

X = data[FEATURES]
y = data["Metabolic_Risk"]

In [15]:
models = {

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),

    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf"))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("model", GradientBoostingClassifier(
            random_state=42
        ))
    ])
}


In [16]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="f1"
    )

    results.append([
        name,
        np.mean(scores),
        np.std(scores)
    ])

results_df = pd.DataFrame(
    results,
    columns=["Model", "CV Mean F1", "CV Std"]
)

results_df.sort_values(by="CV Mean F1", ascending=False)


,Model,CV Mean F1,CV Std
0,Logistic Regression,0.718082,0.020269
2,Random Forest,0.717346,0.019657
3,Gradient Boosting,0.712180,0.027013
1,SVM (RBF),0.698320,0.028440
